# Exoplanet plots

Use a live query to the NASA Exoplanet Archive at IPAC using `astroquery` to gather the most recent data on
confirmed exoplanets to make updated plots of exoplanet properties.

The plots we create are:
 * exoplanet mass vs. orbital semi-major axis, color coded by discovery method
 * exoplanet radius vs. year of discovery for transiting exoplanets
 * orbital eccentricity vs. orbital period

Filenames are tagged with the date of retrieval (e.g., `2025Nov25`).

See https://exoplanetarchive.ipac.caltech.edu/docs/program_interfaces.html for details of the API for the
archive

In [ ]:
%matplotlib inline

import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, LogLocator, NullFormatter
import datetime

# astroquery for the NASA Exoplanet Archive

from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

# suppress nuisance warnings

import warnings
warnings.filterwarnings('ignore',category=UserWarning, append=True)
warnings.filterwarnings('ignore',category=RuntimeWarning, append=True)

## Standard Plot Format

Setup the standard plotting format and make the plot.  

In [ ]:
# graphic aspect ratio = width/height

#aspect = 4.0/3.0 # 4:3 letter
aspect = 16.0/9.0 # wide-screen

#
# Don't change these unless you really need to (we never have)
#
# fPage is the horizontal fraction of the page occupied by the figure, default 1.0
#
# scaleFac is the LaTeX includegraphics scaling in units of \textwidth, default 1.0
#

fPage = 1.0
scaleFac = 0.85

# Text width in inches - don't change, this is defined by the print layout

textWidth = 6.0 # inches

figFmt = 'png'
dpi = 600
plotWidth = dpi*fPage*textWidth
plotHeight = plotWidth/aspect
axisFontSize = 10
labelFontSize = 8
lwidth = 0.5
axisPad = 5
wInches = fPage*textWidth # float(plotWidth)/float(dpi)
hInches = wInches/aspect  # float(plotHeight)/float(dpi)
    
# LaTeX is used throughout for markup of symbols, Times-Roman serif font

plt.rc('text', usetex=True)
plt.rc('font', **{'family':'serif','serif':['Times-Roman'],'weight':'bold','size':'16'})

# Font and line weight defaults for axes

matplotlib.rc('axes',linewidth=lwidth)
matplotlib.rcParams.update({'font.size':axisFontSize})

# axis and label padding

plt.rcParams['xtick.major.pad']=f'{axisPad}'
plt.rcParams['ytick.major.pad']=f'{axisPad}'
plt.rcParams['axes.labelpad'] = f'{axisPad}'

## Retrieve the exoplanet archive data

Use `astroquery` to read the NASA Exoplanet Archive. The `selectPars` parameter lists the data to be retrieved. 
See https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html for the columns in the Planetary Systems Composite Parameters (`pscomppars`) table.

In [ ]:
# astroquery selection parameters

selectPars = f"""
pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,
pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,
pl_rade,pl_radeerr1,pl_radeerr2,
pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,
pl_orbper,pl_orbpererr1,pl_orbpererr2,
pl_insol,pl_insolerr1,pl_insolerr2,
discoverymethod,disc_year
"""

exoData = NasaExoplanetArchive.query_criteria(table="pscomppars",select=selectPars)

# all data together

exSMA = np.array(exoData['pl_orbsmax']) # au
exMass = np.array(exoData['pl_bmasse']) # M_earth
exRadE = np.array(exoData['pl_rade']) # R_earth
discYear = np.array(exoData['disc_year'])
ecc = np.array(exoData['pl_orbeccen'])
period = np.array(exoData['pl_orbper']) # days
insol = np.array(exoData['pl_insol']) # F_earth
               
# indexes by type

method = np.array(exoData['discoverymethod'])

iTransit = np.where(method=='Transit')[0]
iRV = np.where(method=='Radial Velocity')[0]
iML = np.where(method=='Microlensing')[0]
iImg = np.where(method=='Imaging')[0]
iAst = np.where(method=='Astrometry')[0]

# query summary

print(f"Retrieved {len(exSMA)} exoplanets from the NASA Exoplanet Archive")
print(f"  {len(iTransit):4d} transiting planets")
print(f"  {len(iRV):4d} RV planets")
print(f"  {len(iML):4d} microlensing planets")
print(f"  {len(iImg):4d} direct imaging planets")
print(f"  {len(iAst):4d} astrometry planets")

methods = ['Transits','RV','Microlensing','Imaging','Astrometry','Other']
colors = ['purple','green','blue','orange','magenta','black']

# axis limits

aMin = 0.005 # au
aMax = 4500. # au

mMin = 0.03 # M_earth
mMax = 1.5e4 # M_earth

rMin = 0.1 # R_earth
rMax = 50 # R_earth, ~10 R_jup

dyMin = np.min(discYear) - 1.0
dyMax = np.max(discYear) + 1.0

pMin = 0.1 # days
pMax = 20000. # days

# Useful values

M_Earth = 5.97217e24 # kg
M_Jup = 1.89812e27 # kg
M_Sun = 1.98841e30 # kg

Mje = M_Jup/M_Earth

Mbd = 13.6*Mje # minimum brown dwarf mass (deuterium burning limit) in Earth masses, but range because depends in He/H
Mhb = 0.075*M_Sun/M_Earth # hydrogen burning limit from Chabrier 2023, A&A, 672, A119, in Earth masses

# Date of retrieval/plot

now = datetime.datetime.now()
plotDate = now.strftime("%Y%b%d")


## Solar system data

### Planets, Dwarf Planets, and Giant Moons

Planets, dwarf planets, and giant moons are from the JPL Solar System Dynamics database, extracted and organized
into a single CSV file `MassOrbit_Major.csv`.  For this plot we use 4 columns:
 * `Body` - name of the body
 * `a` - orbit semimajor axis in au
 * `ME` - mass in units of M$_E$=5.97271$\times$10$^{24}$ kg
 * `RE` - radius in units of R$_E$=6371 km
 * `Type` - body type code: T = terrestrial planet, G = gas giant, I = ice giant, D = dwarf planet, DC = dwarf planet candidate, GM = giant moon.

In [ ]:
majorFile = 'MassOrbit_Major.xlsx'

data = pd.read_excel(majorFile,comment='#')
bodyName = np.array(data['Body'])
bodyAU = np.array(data['a'])
bodyType = np.array(data['Type'])
bodyMass = np.array(data['ME'])
bodyRadius = np.array(data['RE'])
bodyAU = np.array(data['a'])

# colors etc.

bodySize = {'T':5,'G':8,'I':6,'D':4,'DC':3,'GM':3}
bodyColor = {'T':'#bbbbbb','G':'beige','I':'cyan','D':'snow','DC':'white','GM':'orange'}

## Plot Mass vs orbit semimajor axis

In [ ]:
plotFile = f"exoplanet_Ma_{plotDate}.png"

fig,ax = plt.subplots(figsize=(wInches,hInches),dpi=dpi)

ax.tick_params('both',length=6,width=lwidth,which='major',direction='in',top=True,right=True)
ax.tick_params('both',length=3,width=lwidth,which='minor',direction='in',top=True,right=True)

ax.set_xlim(aMin,aMax)
ax.set_xscale('log')
ax.xaxis.set_major_locator(LogLocator(base=10.0,subs=(1.0,),numticks=100))
ax.xaxis.set_minor_locator(LogLocator(base=10.0,subs=np.arange(2,10)*0.1,numticks=100))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xticks([0.01,0.1,1,10,100,1000])
ax.set_xticklabels(['0.01','0.1','1','10','100','1000'])
ax.set_xlabel(r'Orbit Semimajor Axis [AU]',fontsize=axisFontSize)

ax.set_ylim(mMin,mMax)
ax.set_yscale('log')
ax.yaxis.set_major_locator(LogLocator(base=10.0,subs=(1.0,),numticks=100))
ax.yaxis.set_minor_locator(LogLocator(base=10.0,subs=np.arange(2,10)*0.1,numticks=100))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_yticks([0.1,1,10,100,1e3,1e4])
ax.set_yticklabels(['0.1','1','10','100','10$^3$','10$^4$'])
ax.set_ylabel(r'Mass [M$_{\rm Earth}$]',fontsize=axisFontSize)

# exoplanets

ax.plot(exSMA[iTransit],exMass[iTransit],'o',mfc='#7F3C8D',mec='black',ms=2,mew=0.2,zorder=6,label='Transit')
ax.plot(exSMA[iRV],exMass[iRV],'o',mfc='#11A579',mec='black',ms=2,mew=0.2,zorder=6,label='Radial Velocity')
ax.plot(exSMA[iML],exMass[iML],'o',mfc='#bb0000',mec='black',ms=2,mew=0.2,zorder=6,label='Microlensing') # scarlet, of course
ax.plot(exSMA[iImg],exMass[iImg],'o',mfc='#F2B701',mec='black',ms=2,mew=0.2,zorder=6,label='Imaging')
ax.plot(exSMA[iAst],exMass[iAst],'o',mfc='magenta',mec='black',ms=2,mew=0.2,zorder=6,label='Astrometry')

ax.plot(exSMA,exMass,'o',mfc='black',mec='black',ms=2,mew=0.2,zorder=5,label='Other')
   
# Major bodies (planets, dwarf planets, giant moons)

for i in range(len(bodyName)):
    t = bodyType[i]
    ax.plot(bodyAU[i],bodyMass[i],'o',mfc=bodyColor[t],mec='black',ms=bodySize[t],mew=0.2,zorder=10)

    if t == 'G':
        labelTxt = bodyName[i]
        ax.text(1.25*bodyAU[i],bodyMass[i],labelTxt[0],va='center',ha='left',fontsize=labelFontSize,
                color='black',zorder=10)    
    elif t == 'I':
        labelTxt = bodyName[i]
        if bodyName[i] == 'Uranus':
            ax.text(bodyAU[i],1.2*bodyMass[i],labelTxt[0],va='bottom',ha='center',fontsize=labelFontSize,
                    color='blue',zorder=10)    
        else:
            ax.text(bodyAU[i],bodyMass[i]/1.3,labelTxt[0],va='top',ha='center',fontsize=labelFontSize,
                    color='blue',zorder=10)    
            
    elif t == 'T':
        labelTxt = bodyName[i][0]
        if bodyName[i]=='Mars':
            ax.text(bodyAU[i],1.15*bodyMass[i],'M',va='bottom',ha='center',fontsize=labelFontSize,
                color='black',zorder=10)
        elif bodyName[i]=='Venus' or bodyName[i] == 'Earth':
            ax.text(bodyAU[i],bodyMass[i]/1.2,labelTxt,va='top',ha='center',fontsize=labelFontSize,
                color='black',zorder=10)          
        elif bodyName[i]=='Mercury':
            ax.text(bodyAU[i],1.15*bodyMass[i],'m',va='bottom',ha='center',fontsize=labelFontSize,
                color='black',zorder=10)

# mass limits

# ax.hlines([Mbd,Mhb],aMin,aMax,ls=[':'],colors=['black'],lw=0.5,zorder=6)

# make the plot and hardcopy

leg = ax.legend(loc='lower right',frameon=False,prop={'size':8})
for handle in leg.legend_handles:
    handle.set_markersize(4)
    handle.set_alpha(1.0)

plt.savefig(plotFile,bbox_inches='tight',facecolor='white')

plt.show()

## Plot radius vs. discovery year

In [ ]:
plotFile = f"exoplanet_radDisc_{plotDate}.png"

fig,ax = plt.subplots(figsize=(wInches,hInches),dpi=dpi)

ax.tick_params('both',length=6,width=lwidth,which='major',direction='in',top=True,right=True)
ax.tick_params('both',length=3,width=lwidth,which='minor',direction='in',top=True,right=True)

dyMin = 1994.
ax.set_xlim(dyMin,dyMax)
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
ax.set_xlabel(r'Discovery Year',fontsize=axisFontSize)

ax.set_ylim(rMin,rMax)
ax.set_yscale('log')
ax.yaxis.set_major_locator(LogLocator(base=10.0,subs=(1.0,),numticks=100))
ax.yaxis.set_minor_locator(LogLocator(base=10.0,subs=np.arange(2,10)*0.1,numticks=100))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_yticks([0.1,1,10])
ax.set_yticklabels(['0.1','1','10'])
ax.set_ylabel(r'Radius[R$_{\rm Earth}$]',fontsize=axisFontSize)

# exoplanets

ax.plot(discYear[iTransit],exRadE[iTransit],'o',mfc='#7F3C8D',mec='black',ms=2,mew=0.2,zorder=7,label='Transit')
ax.plot(discYear[iRV],exRadE[iRV],'o',mfc='#11A579',mec='black',ms=2,mew=0.2,zorder=6,label='RV')

#ax.plot(discYear[iTransit],exRadE[iTransit],'o',ms=2,mfc='black',mec='None',mew=0.0,zorder=10)
#ax.grid(True,which='both',lw=0.5,alpha=0.5)

# solar system radii

Rjup = 11.2 # earth radii
Rnep = 3.88 # earth radii
Rmars = 0.5235
Rmerc = 0.3826
Rmoon = 0.2725

ax.hlines([1.0,Rjup,Rnep,Rmars,Rmerc,Rmoon],dyMin,dyMax,ls=[':'],colors=['black'],lw=0.5,zorder=6)

# make the plot and hardcopy

leg = ax.legend(loc='lower left',frameon=False,prop={'size':8})
for handle in leg.legend_handles:
    handle.set_markersize(4)
    handle.set_alpha(1.0)

plt.savefig(plotFile,bbox_inches='tight',facecolor='white')

plt.show()

## Orbit eccentricity vs. orbital period

Replaces an old plot that used to be on the NASA exoplanet archive website.  This looks better.

In [ ]:
plotFile = f"exoplanet_ecc_{plotDate}.png"

fig,ax = plt.subplots(figsize=(wInches,hInches),dpi=dpi)

ax.tick_params('both',length=6,width=lwidth,which='major',direction='in',top=True,right=True)
ax.tick_params('both',length=3,width=lwidth,which='minor',direction='in',top=True,right=True)

pMax = 1e5
ax.set_xlim(pMin,pMax)
ax.set_xscale("log")
ax.xaxis.set_major_locator(LogLocator(base=10.0,subs=(1.0,),numticks=100))
ax.xaxis.set_minor_locator(LogLocator(base=10.0,subs=np.arange(2,10)*0.1,numticks=100))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xticks([0.1,1,10,1e2,1e3,1e4,1e5])
ax.set_xticklabels(['0.1','1','10','100',r'10$^3$',r'10$^4$',r'10$^5$'])
ax.set_xlabel(r'Orbital period [days]',fontsize=axisFontSize)

ax.set_ylim(-0.05,1.05)
ax.yaxis.set_major_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(0.05))
ax.set_ylabel(r'Orbital eccentricity, e',fontsize=axisFontSize)

# exoplanets

ax.plot(period[iTransit],ecc[iTransit],'o',mfc='#7F3C8D',mec='black',ms=2,mew=0.2,zorder=6,label='Transit')
ax.plot(period[iRV],ecc[iRV],'o',mfc='#11A579',mec='black',ms=2,mew=0.2,zorder=6,label='Radial Velocity')
ax.plot(period[iImg],ecc[iImg],'o',mfc='#F2B701',mec='black',ms=2,mew=0.2,zorder=6,label='Imaging')
ax.plot(period[iAst],ecc[iAst],'o',mfc='magenta',mec='black',ms=2,mew=0.2,zorder=6,label='Astrometry')

# solar system planets

Pjup = 4332.820129 # days
eJ = 0.048386
ax.plot(Pjup,eJ,'o',ms=8,mfc='orange',mec='black',mew=0.2,zorder=8)
ax.text(Pjup,-0.075,'J',color='darkorange',ha='center',va='top',fontsize=labelFontSize,zorder=10)

Pearth = 365.25636 # days
eE = 0.01671123
ax.plot(Pearth,eE,'o',ms=4,mfc='cyan',mec='black',mew=0.2,zorder=8)
ax.text(Pearth,-0.075,'E',color='blue',ha='center',va='top',fontsize=labelFontSize,zorder=10)

# make the plot and hardcopy

leg = ax.legend(loc='upper left',frameon=False,prop={'size':8})
for handle in leg.legend_handles:
    handle.set_markersize(4)
    handle.set_alpha(1.0)

plt.savefig(plotFile,bbox_inches='tight',facecolor='white')

plt.show()